# Train LoRA adapters

Edit the configuration below and run the cells in order. This notebook calls the same implementation as `train.py`.

Run from a local clone with `requirements-experiments.txt` installed. Set `DRY_RUN = False` when the configuration is ready.

In [ ]:
from pathlib import Path
import json
import os
import sys

# Set this to the directory containing README.md, train.py, generate.py, and judge.py.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "configs/example.json").is_file(), "Set REPO_ROOT to your Ben-asba clone"
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

## Configuration

Edit the dictionary below or select another JSON config. Paths resolve against `project_root`. Use a new output directory for reruns to preserve the paper inputs. The example includes all 11 harmful training domains and the secure-code control.

In [ ]:
CONFIG_FILE = REPO_ROOT / "configs/example.json"
CONFIG = json.loads(CONFIG_FILE.read_text())
CONFIG["project_root"] = str(REPO_ROOT)

# Examples:
# CONFIG["model"] = "Qwen/Qwen3-4B"
# CONFIG["domains"] = {"incorrect_math": "incorrect_math.jsonl"}
# CONFIG["adapter_dir"] = "adapters/my-model"
# CONFIG["output_dir"] = "output/my-model"

DRY_RUN = True
FORCE = False

## Training settings

Supply the original single-turn training datasets locally in `data_dir`. Assistant prompt tokens and padding are masked out of the loss. LoRA projections are inferred from the model (`q_proj`/`v_proj`, or Phi-3 `qkv_proj`) or set explicitly.

In [ ]:
CONFIG.setdefault("training", {}).update({
    "learning_rate": 2e-4, "rank": 8, "alpha": 16, "dropout": 0.05,
    "batch_size": 4, "gradient_accumulation_steps": 4,
    "epochs": 3, "warmup_steps": 100, "max_length": 512,
    "dtype": "bfloat16", "seed": 42,
})
# For a CPU smoke test use dtype="float32" and a tiny local model.
# CONFIG["training"]["target_modules"] = ["qkv_proj"]
# CONFIG["training"]["load_in_4bit"] = True

## Run

In [ ]:
from emtransfer.config import validate_config
from train import run

run(validate_config(CONFIG), dry_run=DRY_RUN, force=FORCE)